In [1]:
import pandas as pd
import numpy as np

In [2]:
source_path = "../data/processed/favorita_clean.parquet"
df = pd.read_parquet(source_path)
df.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,type_x,cluster,transactions,dcoilwtico,type_y,locale,locale_name,description,transferred
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,NaN,NaN,Holiday,National,Ecuador,Primer dia del ano,False
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,NaN,NaN,Holiday,National,Ecuador,Primer dia del ano,False
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,NaN,NaN,Holiday,National,Ecuador,Primer dia del ano,False
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13,NaN,NaN,Holiday,National,Ecuador,Primer dia del ano,False
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13,NaN,NaN,Holiday,National,Ecuador,Primer dia del ano,False


# Explore data

In [3]:
df["family"].unique()

array(['AUTOMOTIVE', 'BABY CARE', 'BEAUTY', 'BEVERAGES', 'BOOKS',
       'BREAD/BAKERY', 'CELEBRATION', 'CLEANING', 'DAIRY', 'DELI', 'EGGS',
       'FROZEN FOODS', 'GROCERY I', 'GROCERY II', 'HARDWARE',
       'HOME AND KITCHEN I', 'HOME AND KITCHEN II', 'HOME APPLIANCES',
       'HOME CARE', 'LADIESWEAR', 'LAWN AND GARDEN', 'LINGERIE',
       'LIQUOR,WINE,BEER', 'MAGAZINES', 'MEATS', 'PERSONAL CARE',
       'PET SUPPLIES', 'PLAYERS AND ELECTRONICS', 'POULTRY',
       'PREPARED FOODS', 'PRODUCE', 'SCHOOL AND OFFICE SUPPLIES',
       'SEAFOOD'], dtype=object)

In [4]:
df["family"].isnull().sum()

np.int64(0)

In [5]:
df["store_nbr"].unique()

array([ 1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,  2, 20, 21, 22, 23, 24,
       25, 26, 27, 28, 29,  3, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39,  4,
       40, 41, 42, 43, 44, 45, 46, 47, 48, 49,  5, 50, 51, 52, 53, 54,  6,
        7,  8,  9])

In [6]:
df["store_nbr"].isnull().sum()

np.int64(0)

In [7]:
df["date"].unique()

<DatetimeArray>
['2013-01-01 00:00:00', '2013-01-02 00:00:00', '2013-01-03 00:00:00',
 '2013-01-04 00:00:00', '2013-01-05 00:00:00', '2013-01-06 00:00:00',
 '2013-01-07 00:00:00', '2013-01-08 00:00:00', '2013-01-09 00:00:00',
 '2013-01-10 00:00:00',
 ...
 '2017-08-06 00:00:00', '2017-08-07 00:00:00', '2017-08-08 00:00:00',
 '2017-08-09 00:00:00', '2017-08-10 00:00:00', '2017-08-11 00:00:00',
 '2017-08-12 00:00:00', '2017-08-13 00:00:00', '2017-08-14 00:00:00',
 '2017-08-15 00:00:00']
Length: 1684, dtype: datetime64[ns]

In [8]:
df = df.sort_values(["store_nbr", "family", "date"])
df.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,type_x,cluster,transactions,dcoilwtico,type_y,locale,locale_name,description,transferred
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,NaN,NaN,Holiday,National,Ecuador,Primer dia del ano,False
1782,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,Quito,Pichincha,D,13,2111.0,93.14,None,None,None,None,None
3564,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,Quito,Pichincha,D,13,1833.0,92.97,None,None,None,None,None
5346,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,Quito,Pichincha,D,13,1863.0,93.12,None,None,None,None,None
7128,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,Quito,Pichincha,D,13,1509.0,NaN,Work Day,National,Ecuador,Recupero puente Navidad,False


In [9]:
df.shape

(3054348, 17)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3054348 entries, 0 to 3054215
Data columns (total 17 columns):
 #   Column        Dtype         
---  ------        -----         
 0   id            int64         
 1   date          datetime64[ns]
 2   store_nbr     int64         
 3   family        object        
 4   sales         float64       
 5   onpromotion   int64         
 6   city          object        
 7   state         object        
 8   type_x        object        
 9   cluster       int64         
 10  transactions  float64       
 11  dcoilwtico    float64       
 12  type_y        object        
 13  locale        object        
 14  locale_name   object        
 15  description   object        
 16  transferred   object        
dtypes: datetime64[ns](1), float64(3), int64(4), object(9)
memory usage: 419.5+ MB


# =====
# FeastureStore con Feast
# =====

In [48]:
from feast import FeatureStore, Entity, FeatureView, FeatureService, RequestSource, Field, FileSource, ValueType
from feast.types import Int64, String, Float64, UnixTimestamp

from feast.on_demand_feature_view import on_demand_feature_view

# Definir fuente de datos

In [4]:
import os

source_path = os.path.abspath(source_path)
# print(source_path)  # Absolute path to the file
print(os.path.exists(source_path))  # True/False

True


In [5]:
sales_store_source = FileSource(
    name="sales_store_source",
    path=source_path,
    timestamp_field="date",
)

# Definir entidad

In [6]:
store_entity = Entity(
    name="store_entity",
    join_keys=["store_nbr"],
    value_type=ValueType.INT64
)

family_entity = Entity(
    name="family_entity",
    join_keys=["family"],
    value_type=ValueType.STRING
)

# Definir vista de características

## FeatureView Standar

In [7]:
sales_store_view = FeatureView(
    name="sales_store_view",
    entities=[store_entity, family_entity],
    schema=[
        Field(name="sales", dtype=Float64),
        Field(name="onpromotion", dtype=Int64),
        Field(name="transactions", dtype=Float64),
        Field(name="dcoilwtico", dtype=Float64),
        Field(name="type_y", dtype=String, description="holiday type"),
    ],
    online=True,
    source=sales_store_source,
    tags={"project": "sales_store_prediction"},
)

## FeatureView OnDemand (odfv - OnDemanFeatureView)

In [36]:
# Request features disponibles en tiempo real
request_source = RequestSource(
    name="request_data",
    schema=[
        Field(name="sales", dtype=Float64),
        Field(name="onpromotion", dtype=Int64),
        Field(name="date", dtype=Int64),
    ],
)

In [69]:
@on_demand_feature_view(
    sources=[request_source],
    schema=[
        Field(name="day_of_week", dtype=Int64),
        Field(name="month", dtype=Int64),
        Field(name="is_weekend", dtype=Int64),
        Field(name="sales", dtype=Float64),
        Field(name="onpromotion", dtype=Int64),
        Field(name="sales_x_promo", dtype=Float64),
    ],
)
def sales_time_view(features_df: pd.DataFrame) -> pd.DataFrame:
    df = features_df.copy()
    # df = pd.DataFrame()
    
    # Features de calendario
    df["day_of_week"] = pd.to_datetime(features_df["date"]).dt.dayofweek.astype("int64")
    df["month"] = pd.to_datetime(features_df["date"]).dt.month.astype("int64")
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype("int64")
    
    # Interacciones
    df["sales_x_promo"] = features_df["sales"] * features_df["onpromotion"]
    
    return df

# Definir service

## Service para features standar

In [70]:
sales_store_service = FeatureService(
    name="sales_store_service",
    features=[sales_store_view],
)

## Service para features ondemand

In [71]:
sales_time_service = FeatureService(
    name="sales_time_service",
    features=[sales_time_view],
)

# Registrar entidad y vistas en el FeastureStore

In [72]:
fs = FeatureStore(repo_path="../feature_repo/feature_repo")
fs.apply([
    store_entity, family_entity, 
    sales_store_view, sales_time_view,
    sales_store_service, sales_time_service
])

/home/edyn/Documentos/obsidian-doc/Projects/MachineLearningEngineer/Curso2/Proyecto/Repo/sales_store_prediction_segmentation/.venv/lib/python3.12/site-packages/feast/feature_store.py:583: RuntimeWarning: On demand feature view is an experimental feature. This API is stable, but the functionality does not scale well for offline retrieval
  warnings.warn(


# Consumir features

## Consumir feature Standar

In [14]:
df_sample = df[["store_nbr", "family", "date"]].sample(n=1000, random_state=42)
df_sample = df_sample.rename(columns={"date": "event_timestamp"})
df_sample

,store_nbr,family,event_timestamp
668753,23,DAIRY,2014-01-07
1267004,1,BEAUTY,2014-12-05
997442,45,HOME APPLIANCES,2014-07-06
873596,20,LAWN AND GARDEN,2014-05-02
756145,25,HOME AND KITCHEN II,2014-02-25
...,...,...,...
1286174,46,SEAFOOD,2014-12-15
1331683,24,BABY CARE,2015-01-09
563250,13,CELEBRATION,2013-11-09
1541853,20,PLAYERS AND ELECTRONICS,2015-05-07


In [15]:
fs.get_historical_features(
    entity_df=df_sample,
    features=sales_store_service,
    # entity_df=df[["store_nbr", "family", "date"]],
    # features=["sales_store_view:sales", "sales_store_view:onpromotion", "sales_store_view:transactions", "sales_store_view:dcoilwtico", "sales_store_view:type_y"]
).to_df()

,store_nbr,family,event_timestamp,sales,onpromotion,transactions,dcoilwtico,type_y
0,35,LADIESWEAR,2013-01-05 00:00:00+00:00,0.00000,0,723.0,NaN,Work Day
1,45,GROCERY I,2013-01-07 00:00:00+00:00,6629.00000,0,3339.0,93.20,None
2,35,PLAYERS AND ELECTRONICS,2013-01-07 00:00:00+00:00,0.00000,0,568.0,93.20,None
3,27,PREPARED FOODS,2013-01-13 00:00:00+00:00,167.54001,0,1575.0,NaN,None
4,2,HARDWARE,2013-01-13 00:00:00+00:00,0.00000,0,1988.0,NaN,None
...,...,...,...,...,...,...,...,...
995,32,HOME CARE,2017-07-25 00:00:00+00:00,69.00000,3,497.0,47.77,Additional
996,21,LAWN AND GARDEN,2017-08-07 00:00:00+00:00,4.00000,0,1052.0,49.37,None
997,18,DELI,2017-08-08 00:00:00+00:00,138.52400,5,1337.0,49.07,None
998,1,BABY CARE,2017-08-09 00:00:00+00:00,0.00000,0,1766.0,49.59,None


## Consumir features OnDemand

In [41]:
df_filtered = df[df["onpromotion"] != 0]
df_filtered.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,type_x,cluster,transactions,dcoilwtico,type_y,locale,locale_name,description,transferred
819694,810784,2014-04-01,9,CLEANING,1752.00000,3,Quito,Pichincha,B,6,2408.0,99.69,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
819699,810789,2014-04-01,9,GROCERY I,7685.00000,5,Quito,Pichincha,B,6,2408.0,99.69,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
821481,812571,2014-04-02,9,GROCERY I,6481.00000,3,Quito,Pichincha,B,6,2119.0,99.60,None,None,None,None,None
821497,812587,2014-04-02,9,POULTRY,613.71704,1,Quito,Pichincha,B,6,2119.0,99.60,None,None,None,None,None
823256,814346,2014-04-03,9,BREAD/BAKERY,521.00000,1,Quito,Pichincha,B,6,2277.0,100.29,None,None,None,None,None


In [73]:
df_rows = df_filtered[
    ["store_nbr", "family", "date", "sales", "onpromotion"]
    # ["store_nbr", "family", "date"]
].sample(n=10, random_state=42)
df_rows
# df_sample.head()
# df_sample = df[["store_nbr", "family", "date", "sales"]].sample(n=1000, random_state=42)

,store_nbr,family,date,sales,onpromotion
1282012,3,POULTRY,2014-12-13,1349.270,1
2151063,14,MEATS,2016-04-10,229.438,1
1116756,43,BEVERAGES,2014-09-11,2480.000,2
2361852,29,DELI,2016-07-28,183.443,4
1254391,54,POULTRY,2014-11-27,50.346,7
1806365,42,FROZEN FOODS,2015-09-29,53.000,1
2751141,50,PRODUCE,2017-03-02,2410.774,4
2154693,16,MEATS,2016-04-12,48.473,1
1645519,3,CLEANING,2015-07-02,2478.000,4
2989317,34,GROCERY I,2017-07-10,3614.367,59


In [ ]:
# fs.get_historical_features(
#     entity_df=df_rows,
#     # features=sales_store_service
#     features=["sales_store_view:sales", "sales_store_view:onpromotion"]
# ).to_df()

Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.


,store_nbr,family,date,sales,onpromotion,transactions,dcoilwtico,type_y
0,43,BEVERAGES,2014-09-11 00:00:00+00:00,2480.000,2,1078.0,92.89,None
1,54,POULTRY,2014-11-27 00:00:00+00:00,50.346,7,714.0,73.70,None
2,3,POULTRY,2014-12-13 00:00:00+00:00,1349.270,1,3919.0,NaN,None
3,3,CLEANING,2015-07-02 00:00:00+00:00,2478.000,4,3169.0,56.93,None
4,42,FROZEN FOODS,2015-09-29 00:00:00+00:00,53.000,1,807.0,45.24,None
5,14,MEATS,2016-04-10 00:00:00+00:00,229.438,1,1035.0,NaN,None
6,16,MEATS,2016-04-12 00:00:00+00:00,48.473,1,651.0,42.12,Holiday
7,29,DELI,2016-07-28 00:00:00+00:00,183.443,4,940.0,41.13,None
8,50,PRODUCE,2017-03-02 00:00:00+00:00,2410.774,4,2403.0,52.63,Holiday
9,34,GROCERY I,2017-07-10 00:00:00+00:00,3614.367,59,2010.0,44.40,None


In [74]:
entity_rows = df_rows.to_dict(orient="records")
entity_rows

[{'store_nbr': 3,
  'family': 'POULTRY',
  'date': Timestamp('2014-12-13 00:00:00'),
  'sales': 1349.27,
  'onpromotion': 1},
 {'store_nbr': 14,
  'family': 'MEATS',
  'date': Timestamp('2016-04-10 00:00:00'),
  'sales': 229.438,
  'onpromotion': 1},
 {'store_nbr': 43,
  'family': 'BEVERAGES',
  'date': Timestamp('2014-09-11 00:00:00'),
  'sales': 2480.0,
  'onpromotion': 2},
 {'store_nbr': 29,
  'family': 'DELI',
  'date': Timestamp('2016-07-28 00:00:00'),
  'sales': 183.443,
  'onpromotion': 4},
 {'store_nbr': 54,
  'family': 'POULTRY',
  'date': Timestamp('2014-11-27 00:00:00'),
  'sales': 50.346,
  'onpromotion': 7},
 {'store_nbr': 42,
  'family': 'FROZEN FOODS',
  'date': Timestamp('2015-09-29 00:00:00'),
  'sales': 53.0,
  'onpromotion': 1},
 {'store_nbr': 50,
  'family': 'PRODUCE',
  'date': Timestamp('2017-03-02 00:00:00'),
  'sales': 2410.774,
  'onpromotion': 4},
 {'store_nbr': 16,
  'family': 'MEATS',
  'date': Timestamp('2016-04-12 00:00:00'),
  'sales': 48.473,
  'onpromot

In [ ]:
# entity_rows = [
#     {"store_nbr": 1, "family": "GROCERY I", "date": "2017-08-15", "sales": 120, "onpromotion": 5},
#     {"store_nbr": 2, "family": "BEVERAGES", "date": "2025-08-17", "sales": 80, "onpromotion": 3},
# ]

In [75]:
fs.get_online_features(
    entity_rows=entity_rows,
    features=sales_time_service,
    # features=[
    #     "sales_store_view:sales", "sales_store_view:onpromotion", "sales_store_view:transactions", "sales_store_view:dcoilwtico", "sales_store_view:type_y",
    #     "sales_time_view:day_of_week", "sales_time_view:month", "sales_time_view:is_weekend", "sales_time_view:sales_x_promo"
    # ]
).to_df()

,store_nbr,family,sales,onpromotion,day_of_week,month,is_weekend,sales_x_promo
0,3,POULTRY,1349.270,1,5,12,1,1349.270
1,14,MEATS,229.438,1,6,4,1,229.438
2,43,BEVERAGES,2480.000,2,3,9,0,4960.000
3,29,DELI,183.443,4,3,7,0,733.772
4,54,POULTRY,50.346,7,3,11,0,352.422
5,42,FROZEN FOODS,53.000,1,1,9,0,53.000
6,50,PRODUCE,2410.774,4,3,3,0,9643.096
7,16,MEATS,48.473,1,1,4,0,48.473
8,3,CLEANING,2478.000,4,3,7,0,9912.000
9,34,GROCERY I,3614.367,59,0,7,0,213247.653


# FEAST Lineage

![feast_lineage](../docs/feast_lineage.png)